In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

from students.panteeva.lesson3 import Exercise

In [ ]:
digits = load_digits()
X, y = digits.data, digits.target
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

In [ ]:
n_ep = 50
seeds = [100, 200, 300]
lr_values = [0.003, 0.005, 0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0]
batch_sizes = [8, 16, 32, 64, None]
hidden_size = 32

results = []

for lr in lr_values:
    for batch_size in batch_sizes:
        accuracy_scores = []

        for seed in seeds:
            rng = np.random.default_rng(seed)
            model = Exercise.create_model(
                Exercise.create_linear_layer(X_train.shape[1], hidden_size, rng=rng),
                Exercise.create_relu_layer(),
                Exercise.create_linear_layer(hidden_size, len(np.unique(y_train)), rng=rng),
            )
            loss = Exercise.create_cross_entropy_loss()

            batch_size = batch_size if batch_size is not None else len(X_train)
            Exercise.train_model(
                model=model,
                loss=loss,
                x=X_train,
                y=y_train,
                lr=lr,
                n_epoch=n_ep,
                batch_size=batch_size,
            )

            val_predictions = model.forward(X_val)
            val_predicted_classes = np.argmax(val_predictions, axis=1)
            accuracy = np.mean(val_predicted_classes == y_val)

            accuracy_scores.append(accuracy)

        results.append({"lr": lr, "batch_size": batch_size, "accuracy": np.mean(accuracy_scores)})

results.sort(key=lambda x: -x["accuracy"])
print("top-5")
for r in results[:5]:
    batch_size = "None" if r["batch_size"] is None else str(r["batch_size"])
    print(f"{r['lr']:<8.3f} {batch_size:<8} {r['accuracy']:.4f}")

In [ ]:
best = results[0]
best_lr = best["lr"]
best_bs = best["batch_size"]

X_train_full = np.vstack([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])

final_test_accuracies = []
for seed in seeds:
    rng = np.random.default_rng(seed)
    model = Exercise.create_model(
        Exercise.create_linear_layer(X_train_full.shape[1], hidden_size, rng=rng),
        Exercise.create_relu_layer(),
        Exercise.create_linear_layer(hidden_size, len(np.unique(y_train_full)), rng=rng),
    )
    loss = Exercise.create_cross_entropy_loss()

    batch_size = best_bs if best_bs is not None else len(X_train_full)
    Exercise.train_model(
        model=model,
        loss=loss,
        x=X_train_full,
        y=y_train_full,
        lr=best_lr,
        n_epoch=n_ep,
        batch_size=batch_size,
    )

    test_predictions = model.forward(X_test)
    test_predicted_classes = np.argmax(test_predictions, axis=1)
    test_acc = np.mean(test_predicted_classes == y_test)
    final_test_accuracies.append(test_acc)
    print(f"seed {seed}: test_accuracy = {test_acc:.4f}")
print(f"final accuracy: {np.mean(final_test_accuracies):.4f}")